# Scenario: Investigating AI Rejection Rates

In [3]:
import pandas as pd
import sqlite3
# Dataset tracking AI suggestions and doctor decisions
override_data = {
    "session_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "practitioner_id": ["Dr. Wang", "Dr. Li", "Dr. Wang", "Dr. Zhang", "Dr. Li", "Dr. Wang", "Dr. Zhang", "Dr. Li"],
    "tcm_syndrome": ["Liver Qi Stagnation", "Spleen Qi Deficiency", "Liver Qi Stagnation", "Kidney Yin Deficiency", 
                    "Spleen Qi Deficiency", "Liver Qi Stagnation", "Kidney Yin Deficiency", "Spleen Qi Deficiency"],
    "doctor_action": ["Accepted", "Overridden", "Accepted", "Accepted", "Overridden", "Overridden", "Accepted", "Overridden"]
}
# adding dataset to DataFrame
df_override_data = pd.DataFrame(override_data)
# creating slq and save the dataframe
connt = sqlite3.connect(":memory:")
df_override_data.to_sql("overrides", connt, index = False, if_exists = "replace")
# creating function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("******************************** Override Audit Database is ready! **************")

******************************** Override Audit Database is ready! **************


# Calculate Overrides by Syndrome

In [9]:
# query for all data to review
all_data = "SELECT * FROM overrides"
print("******************************** all data to review *****************")
display(run_query(all_data))
print()
# query that counts the total number of sessions and the number of overrides (doctor_action = 'Overridden') for each tcm_syndrome.
syndrome_override = """
SELECT tcm_syndrome, 
       COUNT(*) AS total_sessions,
       SUM(CASE WHEN doctor_action = 'Overridden' THEN 1 ELSE 0 END) AS total_overrides
FROM overrides
GROUP BY tcm_syndrome
"""
print("****************************** overrides by syndrome ***********************")
display(run_query(syndrome_override))


******************************** all data to review *****************


,session_id,practitioner_id,tcm_syndrome,doctor_action
0,1001,Dr. Wang,Liver Qi Stagnation,Accepted
1,1002,Dr. Li,Spleen Qi Deficiency,Overridden
2,1003,Dr. Wang,Liver Qi Stagnation,Accepted
3,1004,Dr. Zhang,Kidney Yin Deficiency,Accepted
4,1005,Dr. Li,Spleen Qi Deficiency,Overridden
5,1006,Dr. Wang,Liver Qi Stagnation,Overridden
6,1007,Dr. Zhang,Kidney Yin Deficiency,Accepted
7,1008,Dr. Li,Spleen Qi Deficiency,Overridden



****************************** overrides by syndrome ***********************


,tcm_syndrome,total_sessions,total_overrides
0,Kidney Yin Deficiency,2,0
1,Liver Qi Stagnation,3,1
2,Spleen Qi Deficiency,3,3


# The Rejection Rate Metric

In [12]:
# query to calculate the percentage of sessions that were overridden for each syndrome.
# Filter the results to show only syndromes where the override rate is greater than 50%.
overriden_rate = """
SELECT tcm_syndrome, 
    COUNT(*) AS total_sessions,
    SUM(CASE WHEN doctor_action = 'Overridden' THEN 1 ELSE 0 END) AS total_overrides,
    (CAST(SUM(CASE WHEN doctor_action = 'Overridden' THEN 1 ELSE 0 END) AS REAL) / COUNT(*)) * 100 AS overridden_percentage
FROM overrides
GROUP BY tcm_syndrome
HAVING overridden_percentage > 50
"""
print("************************************* overridden percentage *********************")
display(run_query(overriden_rate))

************************************* overridden percentage *********************


,tcm_syndrome,total_sessions,total_overrides,overridden_percentage
0,Spleen Qi Deficiency,3,3,100.0
